# 01 — Data Preparation

Convert one dataset's raw archive into the figshare-shaped layout that every downstream notebook expects:

```
Drive/Senior_Project/data/<dataset>/processed/
├── images/<image_id>.png       grayscale uint8 256x256
├── masks/<image_id>.png        binary    uint8 256x256, values {0, 255}
├── metadata.csv                one row per image
├── metadata_summary.csv        per-class summary
└── preprocessing_config.json   the exact knobs this run used
```

**To run a different dataset:** edit `DATASET` in cell 6 and re-run all cells. The notebook dispatches to the right converter via `src.preprocess_utils.get_dataset_converter`. No source-code edits needed — adding a 3rd dataset means a new branch in that dispatch, nothing here changes.

**Supported datasets (current):**
- `figshare` — 3,064 single-slice .mat samples, 233 patients (meningioma / glioma / pituitary)
- `brats2020` — 369 multi-modal glioma cases, pre-extracted 2D slices in .h5; we keep all ~57k slices — tumor-bearing slices are labeled `"glioma"`, empty-mask (no visible tumor) slices are labeled `"no_tumor"`. Extracts FLAIR modality, merges sub-region masks into binary whole-tumor.

**Runtime:** any Colab runtime (no GPU needed). figshare ≈ 3–5 minutes. brats2020 ≈ 15–30 minutes (slower due to ~10× more files written to Drive).

## Cell 1 — Install dependencies

In [ ]:
%pip install -q kagglehub h5py opencv-python-headless pillow numpy pandas matplotlib tqdm

## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

## Cell 3 — Configuration

Adjust these knobs before running. No other cells need editing.

In [ ]:
KAGGLEHUB_DATASET_ID = "awsaf49/brats2020-training-data"
MODALITY             = "flair"       # t1 / t1ce / t2 / flair
KEEP_ONLY_WITH_TUMOR = False         # False = keep all ~57k slices (empty-mask → "no_tumor")
MIN_TUMOR_PIXELS     = 0             # minimum mask pixels to count as tumor-bearing
IMAGE_SIZE           = 256
NORMALIZATION        = "percentile"  # or "minmax"
RANDOM_SEED          = 42

print(f"KAGGLEHUB_DATASET_ID = {KAGGLEHUB_DATASET_ID}")
print(f"MODALITY             = {MODALITY}")
print(f"KEEP_ONLY_WITH_TUMOR = {KEEP_ONLY_WITH_TUMOR}")
print(f"MIN_TUMOR_PIXELS     = {MIN_TUMOR_PIXELS}")
print(f"IMAGE_SIZE           = {IMAGE_SIZE}")
print(f"NORMALIZATION        = {NORMALIZATION}")

## Cell 4 — Resolve output paths

## Cell 5 — Download the raw dataset via KaggleHub

KaggleHub caches the dataset under `/root/.cache/kagglehub/` (Colab's local SSD). It's idempotent — re-running won't re-download if it's already cached. The returned path is the **raw** dataset root; we'll discover files there in the next cell.

In [ ]:
from src.file_utils import dataset_paths, ensure_dir

paths = dataset_paths(DRIVE_ROOT, "brats2020")

for key in ("raw", "processed", "images", "masks",
            "raw_dir", "processed_dir", "images_dir", "masks_dir"):
    if key in paths:
        ensure_dir(paths[key])

def pick_path(*names):
    for n in names:
        if n in paths:
            return paths[n]
    raise KeyError(f"none of {names} found in dataset_paths keys: {sorted(paths.keys())}")

IMAGES_DIR                = pick_path("images", "images_dir")
MASKS_DIR                 = pick_path("masks",  "masks_dir")
METADATA_CSV              = pick_path("metadata_csv")
METADATA_SUMMARY_CSV      = pick_path("metadata_summary_csv")
PREPROCESSING_CONFIG_JSON = pick_path("preprocessing_config_json")

for k, v in paths.items():
    print(f"  {k}: {v}")

In [ ]:
import kagglehub
from pathlib import Path

raw_root = Path(kagglehub.dataset_download(KAGGLEHUB_DATASET_ID))
print(f"raw dataset root: {raw_root}")

# Quick look at the top-level structure
top = sorted([p.name for p in raw_root.iterdir()])
print(f"top-level entries ({len(top)}): {top[:10]}{'...' if len(top) > 10 else ''}")

## Cell 6 — Discover sample files

In [ ]:
from src.preprocess_utils import discover_brats2020_h5_files

sample_paths = discover_brats2020_h5_files(raw_root)
n_candidates = len(sample_paths)
print(f"found {n_candidates:,} BraTS2020 .h5 slices")
print(f"first 3: {[p.name for p in sample_paths[:3]]}")
print(f"last 3:  {[p.name for p in sample_paths[-3:]]}")

# 369 patients × 155 slices = 57,195 total
assert 50_000 <= n_candidates <= 60_000, f"expected ~57,195 brats slices, found {n_candidates}"
print("count sanity: OK")

## Cell 7 — Convert raw `.h5` slices → PNG images + masks + metadata rows

This is the slow cell. Each `convert_brats2020_h5_to_png_record(...)` call returns either:
- a **dict** — the metadata row (kept)
- `None` — the slice was filtered out (only when `KEEP_ONLY_WITH_TUMOR=True`)

In [ ]:
from tqdm.auto import tqdm
from src.preprocess_utils import convert_brats2020_h5_to_png_record

records = []
n_skipped = 0
n_errors  = 0
errors    = []

for src_path in tqdm(sample_paths, desc="converting brats2020"):
    try:
        rec = convert_brats2020_h5_to_png_record(
            src_path,
            image_out_dir=IMAGES_DIR,
            mask_out_dir=MASKS_DIR,
            target_size=(IMAGE_SIZE, IMAGE_SIZE),
            normalization=NORMALIZATION,
            modality=MODALITY,
            keep_only_with_tumor=KEEP_ONLY_WITH_TUMOR,
            min_tumor_pixels=MIN_TUMOR_PIXELS,
            path_style="relative",
            project_root=DRIVE_ROOT,
        )
    except Exception as e:
        n_errors += 1
        if len(errors) < 10:
            errors.append((src_path.name, repr(e)))
        continue

    if rec is None:
        n_skipped += 1
        continue
    records.append(rec)

print(f"\nfound:    {n_candidates:,}")
print(f"kept:     {len(records):,}")
print(f"skipped:  {n_skipped:,}   (empty-tumor filtered out)")
print(f"errors:   {n_errors}")
if errors:
    print("\nfirst errors:")
    for name, err in errors:
        print(f"  {name}: {err}")
assert n_errors == 0, "stop and inspect errors before continuing"
assert len(records) > 0, "no records produced — check the converter"

## Cell 8 — Write `metadata.csv`

In [ ]:
import pandas as pd
from src.data_utils import REQUIRED_METADATA_COLUMNS, validate_metadata

metadata_df = pd.DataFrame.from_records(records)
print(f"metadata_df shape: {metadata_df.shape}")
print(f"columns: {list(metadata_df.columns)}")

validate_metadata(metadata_df)
print(f"\nrequired columns present: {sorted(REQUIRED_METADATA_COLUMNS)}")

metadata_df.to_csv(METADATA_CSV, index=False)
print(f"\nwrote {METADATA_CSV}")
metadata_df.head(3)

## Cell 9 — Write `metadata_summary.csv` (per-class summary)

In [ ]:
from src.data_utils import metadata_summary

summary_df = metadata_summary(metadata_df)
summary_df.to_csv(METADATA_SUMMARY_CSV, index=False)
print(f"wrote {METADATA_SUMMARY_CSV}")
summary_df

## Cell 10 — Write `preprocessing_config.json` (full provenance of this run)

In [ ]:
from src.file_utils import save_json
from datetime import datetime, timezone

preprocessing_config = {
    "dataset":              "brats2020",
    "kagglehub_dataset_id": KAGGLEHUB_DATASET_ID,
    "raw_root":             str(raw_root),
    "image_size":           IMAGE_SIZE,
    "normalization":        NORMALIZATION,
    "modality":             MODALITY,
    "keep_only_with_tumor": KEEP_ONLY_WITH_TUMOR,
    "min_tumor_pixels":     MIN_TUMOR_PIXELS,
    "n_candidates":         int(n_candidates),
    "n_kept":               int(len(records)),
    "n_skipped":            int(n_skipped),
    "n_errors":             int(n_errors),
    "generated_at":         datetime.now(timezone.utc).isoformat(),
}
save_json(preprocessing_config, PREPROCESSING_CONFIG_JSON)
print(f"wrote {PREPROCESSING_CONFIG_JSON}")
print()
for k, v in preprocessing_config.items():
    print(f"  {k}: {v}")

## Cell 11 — Visualize class examples (sanity check)

In [ ]:
from src.vis_utils import show_class_examples

show_class_examples(
    metadata_df=metadata_df,
    project_root=DRIVE_ROOT,
    n_per_class=3,

)

## Done.

You now have, on Drive:

```
data/brats2020/processed/
├── images/<image_id>.png        (~57k slices, FLAIR modality)
├── masks/<image_id>.png
├── metadata.csv                 tumor_class = "glioma" or "no_tumor"
├── metadata_summary.csv
└── preprocessing_config.json
outputs/figures/data_preparation/brats2020/class_examples_grid.png
```

Next: **NB02 (`02_split.ipynb`)** — generate 5-fold CSVs (`patient_level` and `image_level` schemes).